In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/mResearch"
%cd $PROJECT_PATH

Mounted at /content/drive
/content/drive/MyDrive/mResearch


In [ ]:
!pip install -q tensorflow scikit-learn pandas matplotlib tqdm

In [ ]:
import sys
import os

sys.path.append(PROJECT_PATH)

print("✅ Project path added")

✅ Project path added


In [ ]:
from src.config import Config

from experiments.external_validation import main as run_external

In [ ]:
cfg = Config()

# DATA
cfg.TRAIN_PATH = os.path.join(PROJECT_PATH, "datasets/training")
cfg.EXTERNAL_PATH  = os.path.join(PROJECT_PATH, "datasets/testing")

# METRICS
cfg.BOOTSTRAP_SAMPLES = 1000
cfg.CI_ALPHA = 0.95

cfg.set_seed()

print("✅ Config ready")

✅ Config ready


In [ ]:
import numpy as np

# Model 1
m1_path = os.path.join(PROJECT_PATH, "outputs/fusion_model_1 (New)/predictions/external/external_predictions.npz")
m1_int_path = os.path.join(PROJECT_PATH, "outputs/fusion_model_1 (New)/predictions/internal/aggregated_predictions.npz")
m1 = np.load(m1_path)
m1_int = np.load(m1_int_path)

# Model 2
m2_path = os.path.join(PROJECT_PATH, "outputs/fusion_model_2 (New)/predictions/external/external_predictions.npz")
m2_int_path = os.path.join(PROJECT_PATH, "outputs/fusion_model_2 (New)/predictions/internal/aggregated_predictions.npz")
m2 = np.load(m2_path)
m2_int = np.load(m2_int_path)

y_true = m1["y_true"]
y_true_int = m1_int["y_true"]

y_prob_m1 = m1["y_prob"]
y_pred_m1 = m1["y_pred"]

y_prob_m2 = m2["y_prob"]
y_pred_m2 = m2["y_pred"]

y_prob_m1_int = m1_int["y_prob"]
y_pred_m1_int = m1_int["y_pred"]

y_prob_m2_int = m2_int["y_prob"]
y_pred_m2_int = m2_int["y_pred"]

print("✅ Loaded predictions")

✅ Loaded predictions


In [ ]:
from src.evaluation.roc import ROCAnalysis

roc_path = os.path.join(PROJECT_PATH, "outputs/model_comparison_roc(external_new).png")

ROCAnalysis.plot_model_comparison(
    y_true,
    {
        "ES 1": y_prob_m1,
        "ES 2": y_prob_m2
    },
    roc_path
)

print("✅ ROC comparison saved")

✅ ROC comparison saved


In [ ]:
from src.evaluation.roc import ROCAnalysis

roc_path = os.path.join(PROJECT_PATH, "outputs/model_comparison_roc(internal_new).png")

ROCAnalysis.plot_model_comparison(
    y_true_int,
    {
        "ES 1": y_prob_m1_int,
        "ES 2": y_prob_m2_int
    },
    roc_path
)

print("✅ ROC comparison saved")

✅ ROC comparison saved


In [ ]:
from src.evaluation.statistics import Statistics

p_value = Statistics.delong_roc_test(
    y_true,
    y_prob_m1,
    y_prob_m2
)

print(f"\n📊 DeLong Test (Model 1 (External) vs Model 2 (External))")
print(f"p-value: {p_value:.6f}")
print(f"Significance: {Statistics.significance_label(p_value)}")


📊 DeLong Test (Model 1 (External) vs Model 2 (External))
p-value: 0.948399
Significance: ns


In [ ]:
from src.evaluation.statistics import Statistics

p_value_int = Statistics.delong_roc_test(
    y_true_int,
    y_prob_m1_int,
    y_prob_m2_int
)

print(f"\n📊 DeLong Test (Model 1 (Internal) vs Model 2 (Internal))")
print(f"p-value: {p_value_int:.6f}")
print(f"Significance: {Statistics.significance_label(p_value_int)}")


📊 DeLong Test (Model 1 (Internal) vs Model 2 (Internal))
p-value: 0.994166
Significance: ns
